In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/raw/Bank_Churn.csv')
df = df.drop(columns=['CustomerId', 'Surname'])


In [2]:
# String conversion (Gender and Geography)

# Gender - 0(f)/1(m)
df['Gender'] = (df['Gender'] == 'Male').astype(int)
df['Gender'].value_counts() #Check for sure

# Geography - one-hot encoding
df = pd.get_dummies(df, columns=['Geography'], drop_first=True, dtype=int)
print(df.columns)


Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited',
       'Geography_Germany', 'Geography_Spain'],
      dtype='str')


In [3]:
# Derived features - encode domain knowledge
df['BalancePerProduct'] = df['Balance']/df['NumOfProducts']
df['BalanceSalaryRatio'] = df['Balance']/df['EstimatedSalary']
df['TenureAgeRatio'] = df['Tenure']/df['Age']
df['IsZeroBalance'] = (df['Balance'] == 0).astype(int) #Linear models might treat zero balance as the smallest number, this action is to make the distinction explicit.
df['AgeGroup'] = pd.cut(df['Age'], bins=[18, 30, 45, 60, 100], labels=['18-30', '30-45', '45-60', '60+'],right=False,)
df = pd.get_dummies(df, columns=['AgeGroup'], drop_first=True, dtype=int) #This action is needed since linear model cant capture spikes or thresholds. Binning makes it fair for all models.

df.dtypes

CreditScore             int64
Gender                  int64
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Geography_Germany       int64
Geography_Spain         int64
BalancePerProduct     float64
BalanceSalaryRatio    float64
TenureAgeRatio        float64
IsZeroBalance           int64
AgeGroup_30-45          int64
AgeGroup_45-60          int64
AgeGroup_60+            int64
dtype: object

In [4]:
# Scaling numerics
X = df.drop(columns=['Exited'])
y = df['Exited']

# Split first
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.2, random_state=42, stratify=y)

# Scale continuous numeric columns only
cols_to_scale = ['CreditScore', 'Age', 'Tenure', 'Balance','EstimatedSalary',
                 'BalancePerProduct', 'BalanceSalaryRatio','TenureAgeRatio',]

scaler = StandardScaler()
scaler.fit(X_train[cols_to_scale]) # learn from train data only, not leak distribution as a whole

# Transform
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled[cols_to_scale].agg(['mean','std']).round(3)                                   
X_test_scaled[cols_to_scale].agg(['mean','std']).round(3)   

,CreditScore,Age,Tenure,Balance,EstimatedSalary,BalancePerProduct,BalanceSalaryRatio,TenureAgeRatio
mean,-0.012,-0.012,-0.007,0.008,0.031,0.002,-0.014,0.007
std,1.003,0.982,0.992,1.008,1.015,1.003,0.126,1.015


In [5]:
# Save data
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/churn_features.csv', index=False)                                       
X.to_csv('../data/processed/X.csv', index=False) 
y.to_csv('../data/processed/y.csv', index=False)  


In [9]:
# Recheck
import pandas as pd
                                                     
X_check = pd.read_csv('../data/processed/X.csv')   
y_check = pd.read_csv('../data/processed/y.csv').squeeze()   
                                                   
print('X shape:', X_check.shape)             
print('y shape:', y_check.shape)                                             
print('Any NaNs in X?',
X_check.isnull().any().any())       
print('Class balance:', y_check.mean().round(3))                              
print('Duplicate columns?',                      
X_check.columns.duplicated().any())  
print('\nColumns:', X_check.columns.tolist())    

X shape: (10000, 18)
y shape: (10000,)
Any NaNs in X? False
Class balance: 0.204
Duplicate columns? False

Columns: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'BalancePerProduct', 'BalanceSalaryRatio', 'TenureAgeRatio', 'IsZeroBalance', 'AgeGroup_30-45', 'AgeGroup_45-60', 'AgeGroup_60+']
